In [19]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY    = os.environ['OPENAI_API_KEY']
MIELI_SEARCH_KEY  = os.environ['MIELI_SEARCH_KEY']   # ← 추가

print("API 키 로드 완료")
print(f"OpenAI Key:      {OPENAI_API_KEY[:10]}...")
print(f"MeiliSearch Key: {MIELI_SEARCH_KEY[:10]}...")

API 키 로드 완료
OpenAI Key:      sk-proj-00...
MeiliSearch Key: L-F-ISeH_Q...


In [20]:
#기존 캐시된 키 강제 제거
if 'OPENAI_API_KEY' in os.environ:
    del os.environ['OPENAI_API_KEY']

In [21]:
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import load_prompt
from langchain_core.output_parsers import StrOutputParser


In [22]:
!pip install meilisearch

# 마일리서치 인덱스 연결

In [23]:
import meilisearch

In [24]:
import json

# 오류 수정 1: localhostL → localhost (L 오타 제거)
# 오류 수정 2: MIELI_SEARCH_KEY → 위 셀에서 로드한 변수 사용
client = meilisearch.Client('http://localhost:7700', MIELI_SEARCH_KEY)
print("MeiliSearch 클라이언트 연결 완료")

MeiliSearch 클라이언트 연결 완료


In [25]:
client.index('nasdaq')

In [26]:
# 스크리너

In [27]:
# 데이터 로드
import pandas as pd

# CSV 파일 위치: llm/langchain_Proj/ 폴더 안에 있음
df = pd.read_csv('langchain_Proj/nasdaq_screener_1775713077275.csv', na_filter=False)
df.head(3)

,Symbol,Name,Last Sale,Net Change,% Change,Market Cap,Country,IPO Year,Volume,Sector,Industry
0,A,Agilent Technologies Inc. Common Stock,$116.92,3.04,2.669%,33041862904.00,United States,1999,1384565,Industrials,Biotechnology: Laboratory Analytical Instruments
1,AA,Alcoa Corporation Common Stock,$71.76,-1.20,-1.645%,18934772426.00,United States,2016,7131647,Industrials,Aluminum
2,AACB,Artius II Acquisition Inc. Class A Ordinary Sh...,$10.36,0.00,0.00%,0.00,United States,2025,106,,


In [28]:
# 전처리
# 1) Symbol이 영문/숫자/특수문자(.-/)만 포함된 행 필터링
df = df[df['Symbol'].str.contains(r'^[a-zA-Z0-9.\-/^]+$', regex=True)]

# 2) id 컬럼 생성 — 슬래시(/) → 언더바(_) 로 치환 (MeiliSearch id 규칙)
df['id'] = df['Symbol'].str.strip().replace(r'[/^]', '_', regex=True)

# 3) 딕셔너리 리스트로 변환
d = df.to_dict(orient='records')

print(f"전처리 완료 — 총 {len(d)}개 종목")
print(d[:2])  # 샘플 확인

전처리 완료 — 총 7080개 종목
[{'Symbol': 'A', 'Name': 'Agilent Technologies Inc. Common Stock', 'Last Sale': '$116.92', 'Net Change': 3.04, '% Change': '2.669%', 'Market Cap': '33041862904.00', 'Country': 'United States', 'IPO Year': '1999', 'Volume': 1384565, 'Sector': 'Industrials', 'Industry': 'Biotechnology: Laboratory Analytical Instruments', 'id': 'A'}, {'Symbol': 'AA', 'Name': 'Alcoa Corporation Common Stock ', 'Last Sale': '$71.76', 'Net Change': -1.2, '% Change': '-1.645%', 'Market Cap': '18934772426.00', 'Country': 'United States', 'IPO Year': '2016', 'Volume': 7131647, 'Sector': 'Industrials', 'Industry': 'Aluminum', 'id': 'AA'}]


In [29]:
# 인덱스에 데이터 적재
task = client.index('nasdaq').add_documents(d, primary_key='id')
print("데이터 적재 완료")
print(task)

데이터 적재 완료
task_uid=0 index_uid='nasdaq' status='enqueued' type='documentAdditionOrUpdate' enqueued_at=datetime.datetime(2026, 4, 9, 5, 54, 39, 187183)


In [30]:
# 검색 테스트
# client.index('nasdaq').search('apple')   # 영문 회사명 검색
# client.index('nasdaq').search('사과')    # 한글 검색 (시맨틱 미적용 시 결과 없음)
result = client.index('nasdaq').search('ms')
print(f"검색 결과 수: {result['estimatedTotalHits']}")
for hit in result['hits'][:5]:
    print(hit)

검색 결과 수: 0
